# Task 3 -- Contrastive Learning and CLIP

- `Name  : Rana Taqveem Ul Hassan`
- `Roll #: 25280030`

End-to-end implementation of PA_0's Task 3: zero-shot classification with
CLIP on STL-10, exploring the modality gap between image and text
embeddings, and bridging that gap with an orthogonal Procrustes alignment.
Markdown cells before each implementation carry the original assignment
text plus the conceptual/mathematical background needed to understand *why*
each step works, not just *what* it does. `_(answer here)_` placeholders
are left for the required written analysis -- read the concepts, run the
cells, then formalize your own answers.

## Background: Contrastive Learning, InfoNCE, and CLIP

**Why contrastive learning at all.** The starting motivation (the InfoMax
principle) is that a good representation `Z = f(X)` should preserve as much
*mutual information* with the input `X` as possible -- `I(X;Z)` measures how
much knowing one variable reduces uncertainty about the other. Directly
maximizing MI is intractable for high-dimensional data (it requires
estimating densities we don't have). **Contrastive learning sidesteps this**
by turning representation learning into a classification problem: given an
anchor and a set of candidates (one true positive, many negatives), train an
encoder to *identify the positive*. This is provably connected to MI: the
**InfoNCE loss**,

```
L_InfoNCE = -E[ log( f(x_pos, c) / sum_j f(x_j, c) ) ]
```

satisfies `I(x;c) >= log(N) - L_InfoNCE` -- minimizing InfoNCE *maximizes a
lower bound on mutual information*. So "can you pick the right pair out of a
batch of distractors" is a tractable proxy for "does this representation
preserve the information that matters."

**From single-modality to multimodal.** SimCLR/MoCo/BYOL apply this within
one modality (two augmented views of the same image are the positive pair).
**CLIP (Radford et al., 2021)** applies the same InfoNCE-style idea *across
modalities*: an image and its caption are a positive pair; an image paired
with any other caption in the batch is a negative. Concretely, CLIP uses two
separate encoders -- an image encoder and a text encoder -- that map into a
**shared embedding space**. For a batch of `N` (image, text) pairs, CLIP
computes the full `N x N` cosine-similarity matrix between all image and
text embeddings, then applies a symmetric cross-entropy loss: for each row
(image-to-text direction) and each column (text-to-image direction), the
correct pairing should have the highest similarity in its row/column,
everything else in that batch is a negative. The two directions' losses are
averaged. Trained on ~400M (image, text) pairs scraped from the internet,
this produces encoders whose embeddings are meaningfully alignable across
modalities.

**Why this enables zero-shot classification.** Because the training
objective is exactly "does this image's embedding have the highest
similarity with the correct text embedding among a set of candidates," a
trained CLIP can classify images into *any* set of classes never seen during
training, just by comparing the image's embedding against the text
embeddings of candidate class descriptions (e.g. "a photo of a cat"). No
classification head, no fine-tuning:

```
predicted_class = argmax_c cosine_similarity(image_embedding, text_embedding_c)
```

Since CLIP's embeddings are typically L2-normalized before comparison,
cosine similarity reduces to a plain dot product. This is the mechanism
behind every zero-shot experiment below.

In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets
from torch.utils.data import DataLoader
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ---- Colab efficiency setup ----
# Same pattern used throughout this assignment: mount Drive once so the
# dataset, downloaded CLIP weights, and cached embeddings all survive
# session restarts.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_DIR = '/content/drive/MyDrive/ATML/assignment_00/clip'
except ImportError:
    RUN_DIR = './clip'  # not on Colab

os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(f'{RUN_DIR}/embeddings', exist_ok=True)
DATA_ROOT = f'{RUN_DIR}/data'

# DRY_RUN: validate the whole pipeline on a tiny slice before spending real
# GPU time encoding all 8000 STL-10 test images.
DRY_RUN = True
TEST_SUBSET = 200 if DRY_RUN else None   # None = full 8000-image test set
PAIR_SAMPLES = 100                        # Task 3.2/3.3's 50-100 paired samples

print(f"RUN_DIR={RUN_DIR} | DRY_RUN={DRY_RUN} | TEST_SUBSET={TEST_SUBSET}")

In [ ]:
# Official CLIP implementation (spec footnote 13): https://github.com/openai/CLIP
%pip install -q ftfy regex tqdm
%pip install -q git+https://github.com/openai/CLIP.git
%pip install -q umap-learn

import clip
import umap
from scipy.linalg import orthogonal_procrustes

In [ ]:
# ViT-B/32 -- the exact image encoder PA_0's Figure 1 describes (512-d
# shared embedding space). clip.load returns the model AND its matching
# preprocessing transform (resize/crop/normalize) -- pass PIL images straight
# through `preprocess`, same pattern as HF's image processors used elsewhere
# in this assignment.
model, preprocess = clip.load("ViT-B/32", device=DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"CLIP ViT-B/32 loaded: {n_params:,} parameters")
print(f"Image preprocessing pipeline:\n{preprocess}")

## Task 3.1: Zero-Shot Classification on STL-10

(a) Download the STL-10 dataset from Torchvision.
(b) Load OpenAI's CLIP model from the official CLIP implementation.
(c) Evaluate CLIP for zero-shot classification accuracy on STL-10 using the
following prompting techniques:
  i. Plain labels, e.g., "cat".
  ii. Prompted text, e.g., "a photo of a cat".
  iii. More descriptive variants of the prompts.
(d) You may experiment with the prompts on a small subset. However, compare
the accuracies of at least three different prompting strategies on the
**complete test set**.

### Concepts to consider

**Why the wording of the prompt matters at all.** CLIP's text encoder was
trained on real, naturally-occurring captions scraped from the web --
sentences like "a photo of a cat sitting on a windowsill," not bare category
words. A bare word like `"cat"` is out-of-distribution relative to what the
text encoder saw during training, so its embedding may land in a less
well-calibrated region of the shared space. A templated prompt like `"a
photo of a {class}"` is closer to the training distribution's *style*
(explaining why the original CLIP paper reports meaningful accuracy gains
from prompt engineering, without touching any model weights at all -- this
is a real, reproducible effect worth confirming yourself here).

**Why we can evaluate all three strategies from one image-encoding pass.**
The image embeddings don't depend on the prompting strategy at all --
only the *text* side changes. So the expensive step (encoding 8000 test
images) is done exactly once and cached; each prompting strategy only
needs 10 cheap text encodings (one per class) reused against that same
cached image-embedding matrix.

In [ ]:
# STL-10's `transform` is applied to the PIL image the dataset returns --
# CLIP's own `preprocess` (resize/crop/normalize) slots in directly, so the
# tensors this DataLoader yields are already exactly what model.encode_image
# expects.
test_ds_full = datasets.STL10(root=DATA_ROOT, split='test', download=True, transform=preprocess)
classes_stl10 = test_ds_full.classes
print(f"STL-10 classes: {classes_stl10}")
print(f"Full test set size: {len(test_ds_full)}")

if TEST_SUBSET is not None:
    g = torch.Generator().manual_seed(42)
    idx = torch.randperm(len(test_ds_full), generator=g)[:TEST_SUBSET].tolist()
    test_ds = torch.utils.data.Subset(test_ds_full, idx)
else:
    test_ds = test_ds_full

test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
print(f"Evaluating on {len(test_ds)} images (DRY_RUN={DRY_RUN})")

In [ ]:
def extract_image_embeddings(model, loader, cache_path=None):
    """One forward pass per batch through CLIP's image encoder. Embeddings
    are cached to Drive since this is the one genuinely expensive step here
    -- everything downstream (all three prompting strategies, the modality-
    gap analysis, the Procrustes alignment) reuses this same cached matrix."""
    if cache_path and os.path.exists(cache_path):
        data = torch.load(cache_path)
        return data['embeds'], data['labels']

    model.eval()
    all_embeds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="encoding images"):
            images = images.to(DEVICE)
            embeds = model.encode_image(images).float()
            all_embeds.append(embeds.cpu())
            all_labels.append(labels)

    all_embeds = torch.cat(all_embeds)
    all_labels = torch.cat(all_labels)
    if cache_path:
        torch.save({'embeds': all_embeds, 'labels': all_labels}, cache_path)
    return all_embeds, all_labels

cache_tag = f"subset{TEST_SUBSET}" if TEST_SUBSET is not None else "full"
image_embeds, image_labels = extract_image_embeddings(
    model, test_loader, cache_path=f'{RUN_DIR}/embeddings/stl10_test_{cache_tag}.pt'
)
print(f"image_embeds shape: {tuple(image_embeds.shape)}")  # (N, 512) for ViT-B/32

In [ ]:
# Three prompting strategies, from bare label to a richer, more descriptive
# template. All three are applied uniformly across classes here for
# simplicity -- per-class hand-written descriptions (e.g. calling out a
# cat's whiskers specifically) would be a further, optional refinement.
def build_prompts(classnames, strategy):
    if strategy == "plain":
        return [f"{c}" for c in classnames]
    elif strategy == "templated":
        return [f"a photo of a {c}" for c in classnames]
    elif strategy == "descriptive":
        return [f"a high-quality, clear photo of a {c}, prominently visible in the center of the frame" for c in classnames]
    else:
        raise ValueError(strategy)

def zero_shot_accuracy(model, image_embeds, image_labels, classnames, strategy):
    """Text side only -- 10 cheap text encodings reused against the already-
    cached image_embeds matrix (see extract_image_embeddings above)."""
    prompts = build_prompts(classnames, strategy)
    tokens = clip.tokenize(prompts).to(DEVICE)

    with torch.no_grad():
        text_embeds = model.encode_text(tokens).float().cpu()

    # L2-normalize -- CLIP's own similarity computation is always over unit
    # vectors, so cosine similarity reduces to a plain dot product.
    img_n = F.normalize(image_embeds, dim=-1)
    txt_n = F.normalize(text_embeds, dim=-1)

    similarity = img_n @ txt_n.T                 # (N_images, N_classes)
    predictions = similarity.argmax(dim=-1)
    accuracy = (predictions == image_labels).float().mean().item() * 100
    return accuracy, text_embeds, prompts

strategies = ["plain", "templated", "descriptive"]
results_31 = {}
for strategy in strategies:
    acc, txt_emb, prompts = zero_shot_accuracy(model, image_embeds, image_labels, classes_stl10, strategy)
    results_31[strategy] = {"accuracy": acc, "text_embeds": txt_emb, "prompts": prompts}
    print(f"{strategy:12s} | example prompt: {prompts[0]!r:50s} | accuracy: {acc:.2f}%")

### Your answer

**Analytical questions (Task 3.1(d)):** Compare the accuracies of the three
prompting strategies on the complete test set. Which performed best, and
does that match the "templated prompts are closer to CLIP's training
distribution" explanation above? Are there any classes where a more
descriptive prompt helped or hurt disproportionately?

_(answer here)_

## Task 3.2: Exploring the Modality Gap

(a) Use the vision and text encoders within CLIP to extract image and label
embeddings for 50-100 STL-10 samples.
(b) Use a dimensionality-reduction technique such as UMAP or t-SNE to
project the embeddings into a two-dimensional space.
(c) Visualize and compare the distributions of the text and image
embeddings.
(d) Briefly explain your findings:
  - How separated are the two modalities?
  - Does normalization affect the modality gap?
  - Why does CLIP still perform well despite this gap?

### Concepts to consider

**What the "modality gap" is.** Even in a well-trained CLIP model, if you
plot image embeddings and text embeddings together in the shared space,
they don't actually overlap -- they tend to occupy visibly separate regions
("cones"), offset from each other by a roughly consistent gap vector (this
was documented explicitly by Liang et al., *Mind the Gap*, 2022). This is
initially counterintuitive: doesn't the whole point of CLIP's training
objective is to make matching image/text pairs have *high* similarity?

**Why the gap exists despite that.** The key is what InfoNCE actually
optimizes for: within a batch, the correct pairing just needs to have
*higher* similarity than the incorrect pairings -- a **relative ranking**
requirement, not a requirement that positive pairs be *close in an absolute
sense*, let alone that the two modalities' embedding clouds fully overlap.
Two contributing mechanisms usually cited:
- **Cone effect from a shared temperature-scaled softmax:** the learned
  temperature and normalization can push embeddings toward a narrow cone
  rather than spreading isotropically across the full sphere, and there's
  no term in the loss forcing image-cone and text-cone to be the *same*
  cone, only that within-batch correct pairs win their softmax.
  Independently initialized encoder architectures/random seeds naturally
  start in different regions, and training never applies enough direct
  pressure to fully erase that initial separation.
- **The two encoders never directly interact.** Image and text embeddings
  are never compared to each other's *own-modality* neighbors in the loss
  (an image is never contrasted against other images) -- only cross-modal
  comparisons matter, so there's no pressure at all on the *within-modality*
  geometry beyond what's needed for cross-modal ranking.

**Why this doesn't break zero-shot classification.** Classification only
needs `argmax_c similarity(image, text_c)` -- a *relative* comparison across
candidate classes for the *same* image, evaluated entirely within the
image-to-all-texts comparison. A constant (or roughly constant) offset
between the two modalities' overall clouds doesn't change which text
embedding is closest to a given image embedding, as long as the offset
doesn't distort the *relative* ordering across classes. This is exactly the
same "only relative ranking matters" property that let InfoNCE work as a
tractable MI proxy in the first place.

In [ ]:
# Reuse the already-cached image embeddings (no need to re-encode) -- take
# a fixed random subset and pair each image with the text embedding of its
# OWN true class (using the templated prompt strategy). Multiple images of
# the same class legitimately share the same paired text vector; this is
# exactly the "per-sample label embedding" the spec asks for, and it's also
# exactly the (X, Y) pairing Task 3.3's Procrustes alignment needs.
g = torch.Generator().manual_seed(42)
pair_idx = torch.randperm(len(image_embeds), generator=g)[:PAIR_SAMPLES]

pair_image_embeds = image_embeds[pair_idx]                       # (n, 512)
pair_labels = image_labels[pair_idx]
templated_text_embeds = results_31["templated"]["text_embeds"]   # (10, 512), one per class
pair_text_embeds = templated_text_embeds[pair_labels]            # (n, 512) -- gather per-sample

print(f"Paired samples: {pair_image_embeds.shape[0]}")
print(f"pair_image_embeds: {tuple(pair_image_embeds.shape)} | pair_text_embeds: {tuple(pair_text_embeds.shape)}")

In [ ]:
def modality_gap_distance(img_embeds, txt_embeds):
    """Quantitative version of "how separated are the modalities": Euclidean
    distance between the two clouds' centroids, in the same (normalized)
    space used for classification."""
    img_centroid = F.normalize(img_embeds, dim=-1).mean(dim=0)
    txt_centroid = F.normalize(txt_embeds, dim=-1).mean(dim=0)
    return (img_centroid - txt_centroid).norm().item()

def plot_modality_gap(ax, img_embeds, txt_embeds, title):
    combined = torch.cat([img_embeds, txt_embeds], dim=0).numpy()
    modality = np.array(["image"] * len(img_embeds) + ["text"] * len(txt_embeds))

    coords = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(combined)

    for m, color in [("image", "tab:blue"), ("text", "tab:orange")]:
        mask = modality == m
        ax.scatter(coords[mask, 0], coords[mask, 1], s=15, alpha=0.6, label=m, color=color)
    ax.set_title(title)
    ax.legend()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Raw (unnormalized) embeddings -- straight out of the encoders.
plot_modality_gap(axes[0], pair_image_embeds, pair_text_embeds, "Raw embeddings")
gap_raw = modality_gap_distance(pair_image_embeds, pair_text_embeds)

# L2-normalized embeddings -- the space CLIP's own similarity actually uses.
plot_modality_gap(
    axes[1], F.normalize(pair_image_embeds, dim=-1), F.normalize(pair_text_embeds, dim=-1),
    "L2-normalized embeddings"
)

plt.tight_layout()
plt.show()

# modality_gap_distance normalizes internally regardless of what's passed in,
# so there's only one meaningful number here, not a "raw" vs "normalized"
# pair -- what differs between the two plots above is only what UMAP *sees*
# (raw-scale vectors vs. unit vectors), which can still shift the visual
# layout even though the underlying cosine geometry (and this metric) is
# identical either way.
print(f"Centroid-distance modality gap (normalized space): {gap_raw:.4f}")

### Your answer

**Analytical questions (Task 3.2(d)):**
- How separated are the two modalities in your plot(s)? Do image and text
  points form clearly distinct regions, or do they interleave?
- Does normalization visibly affect the modality gap in your two panels?
- Given the "relative ranking is all that matters" explanation above, why
  does CLIP still perform well (per your Task 3.1 results) despite this gap?

_(answer here)_

## Task 3.3: Bridging the Modality Gap

(a) One simple method for aligning the modalities is the **orthogonal
Procrustes transform**. Given two sets of embeddings `X` (image features)
and `Y` (text features), the goal is to find an orthogonal matrix `R` that
minimizes `||XR - Y||_F`, where `||.||_F` is the Frobenius norm. The
closed-form solution involves SVD, but a library implementation is
sufficient for this task.
(b) Pair the STL-10 image embeddings with their corresponding text
embeddings.
(c) Learn the optimal rotation matrix `R` using a library implementation of
Procrustes alignment, such as `scipy.linalg.orthogonal_procrustes`.
(d) Apply the rotation transform to the CLIP embeddings.
(e) Visualize the aligned embeddings using t-SNE or UMAP. How does the
alignment affect the modality gap?
(f) Recompute the classification accuracy using the aligned embeddings and
compare the results with Part 0 (Task 3.1's un-aligned baseline).

### Concepts to consider

**What "orthogonal" buys you here.** An orthogonal matrix `R` satisfies
`R^T R = I` -- geometrically, multiplying by `R` is a pure rotation (and
possibly a reflection), with **no scaling, no shearing**. Critically, an
orthogonal transform preserves vector norms and all pairwise angles/
distances *within* the set it's applied to: `||v @ R|| = ||v||` for any `v`.
This matters because it means the alignment can only *rotate the image
cloud as a rigid body* to better match the text cloud -- it can't cheat by
distorting one modality's internal geometry (e.g. stretching it) to
artificially minimize the objective. Since all our embeddings are already
unit-normalized before this step, applying `R` keeps them unit-normalized
automatically -- no renormalization needed afterward.

**Why this is solvable in closed form (the SVD intuition, not a derivation
you need to reproduce by hand).** For `M = X^T Y`, computing its SVD
`M = U Sigma V^T` gives the optimal rotation as `R = U V^T`. Intuitively:
SVD decomposes the "cross-covariance" between the two embedding sets into
independent rotations for each side plus a scaling; discarding the scaling
(`Sigma`) and keeping only the two rotation components (`U`, `V^T`) gives
exactly the best pure-rotation alignment. You don't need to implement this
by hand -- `scipy.linalg.orthogonal_procrustes(X, Y)` does exactly this and
returns `R` directly.

**Why we fit `R` on only the 50-100 paired samples but apply it to all
8000.** `R` is a single `512 x 512` matrix -- a *global* rotation of the
entire embedding space, not something tied to any specific sample. Once
fit on a representative paired subset, it's a fixed linear transform that
can be applied to *any* image embedding from the same space, including ones
never involved in fitting it. This is analogous to fitting PCA components
on a training subset and then transforming new data with them.

In [ ]:
# Fit R on the same 100 paired, L2-normalized (image, text) embeddings from
# Task 3.2 -- exactly the (X, Y) pairing the spec asks for in 3.3(b).
X_pair = F.normalize(pair_image_embeds, dim=-1).numpy()   # image features
Y_pair = F.normalize(pair_text_embeds, dim=-1).numpy()    # text features

R, scale = orthogonal_procrustes(X_pair, Y_pair)
print(f"R shape: {R.shape} | is orthogonal (R^T R ~= I): {np.allclose(R.T @ R, np.eye(R.shape[0]), atol=1e-4)}")

# Apply the SAME fixed rotation to every image embedding in the full test
# set -- R doesn't depend on which images were used to fit it, only on the
# geometry of the shared embedding space.
image_embeds_n = F.normalize(image_embeds, dim=-1).numpy()
aligned_image_embeds = torch.from_numpy(image_embeds_n @ R).float()
print(f"aligned_image_embeds shape: {tuple(aligned_image_embeds.shape)}")
print(f"norm preserved by rotation: original={np.linalg.norm(image_embeds_n[0]):.4f}, "
      f"aligned={aligned_image_embeds[0].norm().item():.4f}")

In [ ]:
# Task 3.3(f): recompute zero-shot accuracy with the aligned image
# embeddings against the ORIGINAL (unrotated) text embeddings, and compare
# directly against Task 3.1's baseline for the same prompting strategy.
def zero_shot_accuracy_aligned(aligned_img_embeds, image_labels, text_embeds):
    txt_n = F.normalize(text_embeds, dim=-1)
    similarity = aligned_img_embeds @ txt_n.T   # aligned_img_embeds already unit-norm (rotation preserves norm)
    predictions = similarity.argmax(dim=-1)
    return (predictions == image_labels).float().mean().item() * 100

print(f"{'strategy':12s} {'baseline (Part 0)':>20s} {'aligned':>12s} {'delta':>10s}")
for strategy in strategies:
    txt_emb = results_31[strategy]["text_embeds"]
    acc_aligned = zero_shot_accuracy_aligned(aligned_image_embeds, image_labels, txt_emb)
    acc_baseline = results_31[strategy]["accuracy"]
    print(f"{strategy:12s} {acc_baseline:19.2f}% {acc_aligned:11.2f}% {acc_aligned - acc_baseline:+9.2f}%")

In [ ]:
# Re-visualize the same paired subset after alignment, and recompute the
# quantitative gap metric -- direct before/after comparison against Task 3.2.
# Alignment is applied per-row independently, so selecting the paired subset
# before or after applying R gives the same result -- just index into the
# already-computed full aligned_image_embeds rather than recomputing.
pair_image_embeds_aligned = aligned_image_embeds[pair_idx]

fig, ax = plt.subplots(figsize=(8, 7))
plot_modality_gap(ax, pair_image_embeds_aligned, F.normalize(pair_text_embeds, dim=-1), "Aligned embeddings (after Procrustes)")
plt.show()

gap_before = modality_gap_distance(pair_image_embeds, pair_text_embeds)
gap_after = modality_gap_distance(pair_image_embeds_aligned, pair_text_embeds)
print(f"Modality gap (centroid distance) before alignment: {gap_before:.4f}")
print(f"Modality gap (centroid distance) after alignment:  {gap_after:.4f}")

### Your answer

**Analytical questions (Task 3.3(e)/(f)):** How does the Procrustes
alignment affect the modality gap, visually and in the quantitative
centroid-distance metric? Does the recomputed zero-shot accuracy improve,
stay flat, or get worse compared to Task 3.1's baseline, for each prompting
strategy -- and does that outcome match what you'd predict from the "only
relative ranking matters for classification" explanation in Task 3.2? If
alignment barely changes accuracy despite visibly shrinking the gap in the
plot, what does that tell you about the relationship between the modality
gap and zero-shot performance?

_(answer here)_